<a href="https://colab.research.google.com/github/taichi0520-hub/my-research-project/blob/main/src/LiveCell%E8%BF%BD%E5%8A%A0%E5%AD%A6%E7%BF%92.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- 1. ライブラリのインストール ---
!pip install "cellpose[gui]" --quiet

import os
import glob
import shutil
import numpy as np
import matplotlib.pyplot as plt
from cellpose import models, io, train
from google.colab import files
from zipfile import ZipFile

# --- 2. データのアップロード ---
print("👇 下のボタンから 'train_data.zip' をアップロードしてください")
uploaded = files.upload()

# アップロードされたZIPを解凍
zip_name = next(iter(uploaded))
with ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall()
    print("✅ 解凍完了")

# データの確認
# _maskがついているファイルをマスク、ついていないものを元画像として認識させます
train_dir = "train_data" # ZIPを解凍してできたフォルダ名に合わせてください
output_dir = "my_custom_models" # モデルの保存先

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

print(f"📂 データの場所: {train_dir}")

In [ ]:
# --- 3. 学習の実行（修正版） ---
# 必要なライブラリのインストール
!pip install scikit-image --quiet

import os
import glob
from skimage.io import imread
from cellpose import models, io, train # trainをimportに追加

# データの確認
train_dir = "train_data" # ZIP解凍後のフォルダ名
output_dir = "my_custom_models"

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 1. 確実な画像とマスクの読み込み
print("⏳ データを読み込んでいます...")
image_paths = []
mask_paths = []

# 対応する拡張子を網羅的に取得
for ext in ('*.png', '*.jpg', '*.jpeg', '*.tif', '*.tiff'):
    for f in glob.glob(os.path.join(train_dir, ext)):
        # マスクファイル自体はスキップ
        if "_mask" in f:
            continue

        # 元画像ファイル名からマスクファイル名を推測
        base, ext_str = os.path.splitext(f)
        mask_f = base + "_mask" + ext_str

        if os.path.exists(mask_f):
            image_paths.append(f)
            mask_paths.append(mask_f)

# NumPy配列としてロード
images = [imread(f) for f in image_paths]
masks = [imread(f) for f in mask_paths]

# 読み込みの整合性検証
if len(images) == 0:
    raise ValueError("❌ 画像が見つかりません。train_dirのパスやファイル名（_maskの有無）を確認してください。")
if len(images) != len(masks):
    raise ValueError(f"❌ 画像とマスクのペア数が不一致です。画像: {len(images)}枚, マスク: {len(masks)}枚")

print(f"✅ {len(images)} ペアの画像を読み込みました。")

# 2. モデルの初期化と学習
model = models.CellposeModel(gpu=True, model_type='cyto2')

n_epochs = 300
channels = [0, 0] # グレースケール画像の場合は [0,0]。RGBで赤に細胞がある場合は [1,0] に変更してください。

print("🚀 追加学習を開始します（数分〜数十分かかります）...")

new_model_path = None # Initialize to None

try:
    # CellposeModel.train ではなく train.train_seg を使用して学習を実行
    new_model_path = train.train_seg(
        model.net, # model.netを渡す
        train_data=images,
        train_labels=masks,
        # channels=channels, # 'channels'引数は不要なためコメントアウト
        save_path=output_dir,
        n_epochs=n_epochs,
        learning_rate=0.0001,
        weight_decay=0.0001,
        nimg_per_epoch=8,
        min_train_masks=1,
        model_name='my_finetuned_model'
    )
    print(f"🎉 学習完了！ モデルが保存されました: {new_model_path}")

except Exception as e:
    print(f"⚠️ 学習中にエラーが発生しました: {e}")
finally:
    # デバッグ情報として、保存パスとディレクトリの内容を出力します
    print(f"Path returned by train.train_seg: {new_model_path}")
    if os.path.exists(output_dir):
        print(f"Contents of {output_dir}: {os.listdir(output_dir)}")
    else:
        print(f"Output directory {output_dir} does not exist.")

### 3.1 モデル保存状況の確認

学習が完了したとされる`my_custom_models`ディレクトリと、その中の`models`サブディレクトリの内容を確認します。これにより、モデルがどこに保存されたか、あるいは保存されなかったかを把握できます。

In [ ]:
import os

output_dir = "my_custom_models"
model_files_dir = os.path.join(output_dir, 'models')

print(f"--- {output_dir} の内容 ---")
if os.path.exists(output_dir):
    print(os.listdir(output_dir))
else:
    print(f"ディレクトリ '{output_dir}' は存在しません。")

print(f"\n--- {model_files_dir} の内容 ---")
if os.path.exists(model_files_dir):
    print(os.listdir(model_files_dir))
else:
    print(f"ディレクトリ '{model_files_dir}' は存在しません。")

print("\nこれらのリストを確認し、モデルファイル（例: 'my_finetuned_model_日付_時刻'で始まるファイル）が存在するかどうかを確認してください。")

In [ ]:
# --- 4. 作成されたモデルのダウンロード ---
import os
import re # For regex to extract base name
from google.colab import files
from zipfile import ZipFile

output_dir = "my_custom_models" # モデルの保存先を再定義
model_files_dir = os.path.join(output_dir, 'models') # 実際のモデルファイルが保存されている場所

print(f"DEBUG: Listing contents of {output_dir}: {os.listdir(output_dir)}") # デバッグ情報

# Check if the model_files_dir exists
if not os.path.exists(model_files_dir):
    print(f"❌ モデルファイルが保存されるべきディレクトリ ({model_files_dir}) が見つかりませんでした。")
    print("学習が正常に終了したか確認してください。")
else:
    print(f"DEBUG: Listing contents of {model_files_dir}: {os.listdir(model_files_dir)}") # デバッグ情報

    # Regular expression to match the base model name with timestamp
    # e.g., 'my_finetuned_model_2024_03_30_13_40_46.079219'
    model_name_pattern = re.compile(r'^(my_finetuned_model_\d{4}_\d{2}_\d{2}_\d{2}_\d{2}_\d{2}\.\d{6})')

    latest_model_base_name = None
    latest_timestamp = None

    # Iterate through files in model_files_dir to find the latest model base name
    for entry in os.listdir(model_files_dir):
        match = model_name_pattern.match(entry)
        if match:
            current_base_name = match.group(1)
            # Extract timestamp part for comparison
            timestamp_str = current_base_name.replace('my_finetuned_model_', '')
            # String comparison works for YYYY_MM_DD...
            if latest_timestamp is None or timestamp_str > latest_timestamp:
                latest_timestamp = timestamp_str
                latest_model_base_name = current_base_name

    if latest_model_base_name:
        files_to_zip = []
        # Collect all files associated with the latest model base name
        for entry in os.listdir(model_files_dir):
            if entry.startswith(latest_model_base_name):
                files_to_zip.append(os.path.join(model_files_dir, entry))

        if files_to_zip:
            zip_filename = f"{latest_model_base_name}.zip"
            with ZipFile(zip_filename, 'w') as zipf:
                for file_path in files_to_zip:
                    zipf.write(file_path, os.path.basename(file_path))
            print(f"💾 ダウンロード中: {zip_filename}")
            files.download(zip_filename)
        else:
            print("❌ モデルファイルが見つかりませんでした。学習が正常に終了したか確認してください。")
    else:
        print("❌ モデルファイルが見つかりませんでした。学習が正常に終了したか確認してください。")